In [ ]:
# preparing dataset 
from datasets import load_dataset, Dataset 



{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [15]:
# preprocessing/Tokenization 
from transformers import AutoTokenizer 

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
dataset = load_dataset('imdb')

# define tokenization function 

def preprocess_function(input):
    return tokenizer(
        input['text'],
        padding = 'max_length',
        truncation= True,
        max_length = 512,
        return_tensors = None 
    )

In [16]:
# apply to entire dataset 
tokenize_datasets = dataset.map(
    preprocess_function, 
    batched = True,
    remove_columns = ['text'],
    num_proc = 4,
)

Map (num_proc=4):   0%|          | 0/25000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/25000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/50000 [00:00<?, ? examples/s]

In [19]:
# fine-tuning with Trainer API 

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import load_dataset 
import numpy as np 
from sklearn.metrics import accuracy_score, f1_score

In [2]:
# step 1: Load model and tokenizer 
model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels =2 # binary classification 
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# step 2: load and preprocess data 
dataset = load_dataset('imdb')

def preprocess_function(input):
    return tokenizer(
        input['text'],
        padding = 'max_length',
        truncation= True,
        max_length = 512,
        return_tensors = None 
    )

tokenize_datasets = dataset.map(
    preprocess_function, 
    batched = True,
    remove_columns = ['text'],
)


# step 3 : defining training arguments


NameError: name 'AutoModelForSequenceClassification' is not defined

In [1]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Step 1: Load model and tokenizer
model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2  # Binary classification
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Step 2: Load and preprocess data
dataset = load_dataset("imdb")

def preprocess_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=['text']
)

# Step 3: Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    
    # Learning
    learning_rate=2e-5,         # Lower than training from scratch
    num_train_epochs=3,         # Usually 2-4 is enough
    weight_decay=0.01,          # L2 regularization
    
    # Batch size
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,  # Simulate larger batch
    
    # Optimization
    warmup_steps=500,           # Gradually increase learning rate
    warmup_ratio=0.1,           # Or use percentage
    
    # Evaluation & Saving
    eval_strategy="epoch",      # Evaluate after each epoch
    save_strategy="epoch",      # Save after each epoch
    load_best_model_at_end=True,  # Keep best checkpoint
    
    # Logging
    logging_steps=100,
    logging_dir='./logs',
    
    # Other
    seed=42,
    push_to_hub=False,          # Push to Hub after training
)

# Step 4: Define metrics
def compute_metrics(eval_pred):
    """
    Computes accuracy, precision, recall, F1
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)
    
    return {
        'accuracy': accuracy,
        'f1': f1
    }

# Step 5: Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

# Step 6: Train!
trainer.train()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


NameError: name 'AutoTokenizer' is not defined